# Data Preprocessing

## Objective

The objective of this notebook is to prepare the Ames Housing dataset for machine learning by performing the following tasks:

- Separate features and target variable
- Split the data into training and testing sets
- Identify numerical and categorical features
- Handle missing values
- Encode categorical variables
- Scale numerical features
- Build a reusable preprocessing pipeline
- Save the preprocessing pipeline for future inference

The output of this notebook will be preprocessed training and testing datasets ready for model training.

In [1]:
# ==========================================
# Import Libraries
# ==========================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

import joblib

In [2]:
# ==========================================
# Load Dataset
# ==========================================

train_df = pd.read_csv("../data/processed/train_feature_engineered.csv")

In [3]:
# ==========================================
# Separate Features and Target
# ==========================================

X = train_df.drop("SalePrice", axis = 1)

y = train_df["SalePrice"]

In [4]:
print(X.shape)
print(y.shape)

(1460, 92)
(1460,)


In [5]:
# ==========================================
# Train Test Split
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

print("Training Features :", X_train.shape)
print("Testing Features :", X_test.shape)

print("Training Labels :", y_train.shape)
print("Testing Labels :", y_test.shape)

Training Features : (1168, 92)
Testing Features : (292, 92)
Training Labels : (1168,)
Testing Labels : (292,)


In [6]:
# ==========================================
# Create Copies
# ==========================================

X_train_processed = X_train.copy()
X_test_processed = X_test.copy()

In [7]:
# These columns don't actually have missing values—they indicate the house doesn't have that feature.

none_cols = [
    "PoolQC",
    "MiscFeature",
    "Alley",
    "Fence",
    "FireplaceQu",
    "GarageType",
    "GarageFinish",
    "GarageQual",
    "GarageCond",
    "BsmtQual",
    "BsmtCond",
    "BsmtExposure",
    "BsmtFinType1",
    "BsmtFinType2",
    "MasVnrType"
]

X_train_processed[none_cols] = X_train_processed[none_cols].fillna("None")
X_test_processed[none_cols] = X_test_processed[none_cols].fillna("None")

In [8]:
# ==========================================
# Numerical Columns
# ==========================================

numerical_features = X_train_processed.select_dtypes(
    include=["int64", "float64"]
).columns

In [9]:
# Fill Numerical columns

for col in numerical_features:

    median = X_train_processed[col].median()

    X_train_processed[col] = X_train_processed[col].fillna(median)

    X_test_processed[col] = X_test_processed[col].fillna(median)

In [10]:
# ==========================================
# Categorical Columns
# ==========================================

categorical_features = X_train_processed.select_dtypes(
    include="object"
).columns

In [11]:
# Fill Categorical columns

for col in categorical_features:

    mode = X_train_processed[col].mode()[0]

    X_train_processed[col] = X_train_processed[col].fillna(mode)

    X_test_processed[col] = X_test_processed[col].fillna(mode)

In [12]:
print(
    "Missing Values in Train :",
    X_train_processed.isnull().sum().sum()
)

print(
    "Missing Values in Test :",
    X_test_processed.isnull().sum().sum()
)

Missing Values in Train : 0
Missing Values in Test : 0


In [13]:
# ==========================================
# One Hot Encoding
# ==========================================

# Ensure test set has the exact same categories as train set to avoid misalignment
for col in categorical_features:
    categories = X_train_processed[col].unique()
    X_train_processed[col] = pd.Categorical(X_train_processed[col], categories=categories)
    X_test_processed[col] = pd.Categorical(X_test_processed[col], categories=categories)

X_train_processed = pd.get_dummies(
    X_train_processed,
    drop_first=True
)

X_test_processed = pd.get_dummies(
    X_test_processed,
    drop_first=True
)

In [14]:
# Align columns

X_train_processed, X_test_processed = X_train_processed.align(
    X_test_processed,
    join="left",
    axis=1,
    fill_value=0
)

In [15]:
# ==========================================
# Feature Scaling
# ==========================================

# Keep encoded (unscaled) data
X_train_encoded = X_train_processed.copy()
X_test_encoded = X_test_processed.copy()

# Create scaled copies
scaler = StandardScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_encoded),
    columns=X_train_encoded.columns,
    index=X_train_encoded.index
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test_encoded),
    columns=X_test_encoded.columns,
    index=X_test_encoded.index
)

In [16]:
print("Training Shape :", X_train_scaled.shape)
print("Testing Shape  :", X_test_scaled.shape)

print()

print(
    "Missing Values in Train :",
    X_train_scaled.isnull().sum().sum()
)

print(
    "Missing Values in Test :",
    X_test_scaled.isnull().sum().sum()
)

Training Shape : (1168, 270)
Testing Shape  : (292, 270)

Missing Values in Train : 0
Missing Values in Test : 0


In [17]:
# Save encoded data
joblib.dump(X_train_encoded, "../models/X_train_encoded.pkl")
joblib.dump(X_test_encoded, "../models/X_test_encoded.pkl")

# Save scaled data
joblib.dump(X_train_scaled, "../models/X_train_scaled.pkl")
joblib.dump(X_test_scaled, "../models/X_test_scaled.pkl")

# Save target
joblib.dump(y_train, "../models/y_train.pkl")
joblib.dump(y_test, "../models/y_test.pkl")

# Save scaler
joblib.dump(scaler, "../models/scaler.pkl")

['../models/scaler.pkl']

# Conclusion

In this notebook, we prepared the dataset for machine learning by:

- Splitting the data into training and testing sets.
- Handling missing values in numerical and categorical features.
- Encoding categorical variables using One-Hot Encoding.
- Scaling numerical features using StandardScaler.
- Ensuring both training and testing datasets have the same feature space.

The processed datasets are now ready for model training.